In [1]:
import pandas as pd
from sklearn.naive_bayes import MultinomialNB
from sklearn.feature_extraction.text import TfidfVectorizer

In [3]:
# !unzip train_essays.csv.zip

Archive:  train_essays.csv.zip
  inflating: train_essays.csv        


In [4]:
import pandas as pd

train_df = pd.read_csv('train_essays.csv')

train_df

,id,prompt_id,text,generated
0,0059830c,0,Cars. Cars have been around since they became ...,0
1,005db917,0,Transportation is a large necessity in most co...,0
2,008f63e3,0,"""America's love affair with it's vehicles seem...",0
3,00940276,0,How often do you ride in a car? Do you drive a...,0
4,00c39458,0,Cars are a wonderful thing. They are perhaps o...,0
...,...,...,...,...
1373,fe6ff9a5,1,There has been a fuss about the Elector Colleg...,0
1374,ff669174,0,Limiting car usage has many advantages. Such a...,0
1375,ffa247e0,0,There's a new trend that has been developing f...,0
1376,ffc237e9,0,As we all know cars are a big part of our soci...,0


In [5]:
train_df.rename(columns={'generated': 'label'}, inplace=True)

In [6]:
train_df['label'].value_counts()

0    1375
1       3
Name: label, dtype: int64

In [7]:
train_df.drop(['id', 'prompt_id'], axis=1, inplace=True)
train_df

,text,label
0,Cars. Cars have been around since they became ...,0
1,Transportation is a large necessity in most co...,0
2,"""America's love affair with it's vehicles seem...",0
3,How often do you ride in a car? Do you drive a...,0
4,Cars are a wonderful thing. They are perhaps o...,0
...,...,...
1373,There has been a fuss about the Elector Colleg...,0
1374,Limiting car usage has many advantages. Such a...,0
1375,There's a new trend that has been developing f...,0
1376,As we all know cars are a big part of our soci...,0


In [8]:
# !unzip train_drcat_02.csv.zip

Archive:  train_drcat_02.csv.zip
  inflating: train_drcat_02.csv      


In [9]:
train = pd.read_csv("train_drcat_02.csv")
train['label'].value_counts()
train.drop(['essay_id', 'source', 'prompt', 'fold'], axis=1, inplace=True)
# train
# train1 = train[train.RDizzl3_seven == False].reset_index(drop=True)
train1=train[train["label"]==1].sample(1300)
# train = train[train.RDizzl3_seven == True].reset_index(drop=True)

train_df=pd.concat([train_df,train1])

# df = df.drop(['prompt_name','source','RDizzl3_seven'],axis = 1)

In [10]:
def clean(data, col):  # Replace each occurrence of pattern/regex in the Series/Index

    # Clean some punctutations
    data[col] = data[col].str.replace('\n', ' \n ')
    data[col] = data[col].str.replace(r'([a-zA-Z]+)([/!?.])([a-zA-Z]+)',r'\1 \2 \3')
    # Replace repeating characters more than 3 times to length of 3
    data[col] = data[col].str.replace(r'([*!?\'])\1\1{2,}',r'\1\1\1')
    # Add space around repeating characters
    data[col] = data[col].str.replace(r'([*!?\']+)',r' \1 ')
    # patterns with repeating characters
    data[col] = data[col].str.replace(r'([a-zA-Z])\1{2,}\b',r'\1\1')
    data[col] = data[col].str.replace(r'([a-zA-Z])\1\1{2,}\B',r'\1\1\1')
    data[col] = data[col].str.replace(r'[ ]{2,}',' ').str.strip()

    return data  # the function returns the processed value

In [11]:
train_df = clean(train_df,'text')

<ipython-input-10-7570c29269b8>:5: FutureWarning: The default value of regex will change from True to False in a future version.
  data[col] = data[col].str.replace(r'([a-zA-Z]+)([/!?.])([a-zA-Z]+)',r'\1 \2 \3')
<ipython-input-10-7570c29269b8>:7: FutureWarning: The default value of regex will change from True to False in a future version.
  data[col] = data[col].str.replace(r'([*!?\'])\1\1{2,}',r'\1\1\1')
<ipython-input-10-7570c29269b8>:9: FutureWarning: The default value of regex will change from True to False in a future version.
  data[col] = data[col].str.replace(r'([*!?\']+)',r' \1 ')
<ipython-input-10-7570c29269b8>:11: FutureWarning: The default value of regex will change from True to False in a future version.
  data[col] = data[col].str.replace(r'([a-zA-Z])\1{2,}\b',r'\1\1')
<ipython-input-10-7570c29269b8>:12: FutureWarning: The default value of regex will change from True to False in a future version.
  data[col] = data[col].str.replace(r'([a-zA-Z])\1\1{2,}\B',r'\1\1\1')
<ipyt

In [12]:
train_df = train_df.sample(frac=1, random_state=1)
train_df.reset_index(drop=True, inplace=True)

split_index_1 = int(len(train_df) * 0.7)
split_index_2 = int(len(train_df) * 0.85)

train_df, val_df, test_df = train_df[:split_index_1], train_df[split_index_1:split_index_2], train_df[split_index_2:]

len(train_df), len(val_df), len(test_df)

(1874, 402, 402)

In [13]:
from sklearn.feature_extraction.text import CountVectorizer
from nltk.tokenize import RegexpTokenizer
#tokenizer to remove unwanted elements from out data like symbols and numbers
token = RegexpTokenizer(r'[a-zA-Z0-9]+')
cv = CountVectorizer(lowercase=True,stop_words='english',ngram_range = (1,1),tokenizer = token.tokenize)
text_counts= cv.fit_transform(train_df['text'])

/usr/local/lib/python3.10/dist-packages/sklearn/feature_extraction/text.py:528: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


In [14]:
text_counts.shape

(1874, 16541)

In [15]:
from sklearn.feature_extraction.text import TfidfTransformer
tf = TfidfTransformer()
total_tfidf= tf.fit_transform(text_counts)
total_tfidf.shape

(1874, 16541)

In [16]:
from sklearn.naive_bayes import MultinomialNB
model = MultinomialNB()
# from sklearn.linear_model import Ridge
# model = Ridge()
model.fit(total_tfidf, train_df['label'])

MultinomialNB()

In [ ]:
# model.score(total_tfidf, total_data['y'])

0.7326381541039744

In [17]:
# val_data = pd.read_csv("/validation_data.csv")
# val_data = clean(val_data,'less_toxic')
# val_data = clean(val_data,'more_toxic')
test_texts = tf.transform(cv.transform(test_df['text']))
# more_toxic = tf.transform(cv.transform(val_data['more_toxic']))
preds = model.predict(test_texts)



In [18]:
preds

array([1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0,
       0, 0, 0, 0, 1, 1, 0, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0, 0, 1, 1, 1, 0,
       0, 0, 1, 0, 1, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 0, 1,
       1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 1,
       0, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1,
       1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0,
       1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0,
       0, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 0, 0, 1, 0, 0, 0, 1,
       0, 0, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 0, 0, 0, 0, 0, 1,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0,
       1, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1,

In [19]:
from sklearn.metrics import classification_report

print(classification_report(test_df['label'], preds))

              precision    recall  f1-score   support

           0       0.93      1.00      0.97       209
           1       1.00      0.92      0.96       193

    accuracy                           0.96       402
   macro avg       0.97      0.96      0.96       402
weighted avg       0.97      0.96      0.96       402

